# Qwen3-Embedding-8B on Colab

Embeds `unique_statements_cleaned.csv` with `Qwen/Qwen3-Embedding-8B` and saves `embeddings_8b.npy`.

1. Runtime > Change runtime type > GPU (A100 or L4 on Colab Pro, T4 on free tier).
2. Open the Files panel (folder icon, left side) and drag in `unique_statements_cleaned.csv`.
3. Run all cells. When `embeddings_8b.npy` appears in the Files panel, download it right away: runtime files are deleted when the session ends.

In [ ]:
!pip install -q -U sentence-transformers transformers bitsandbytes accelerate

In [ ]:
from pathlib import Path

# Files dragged into the Files panel land in /content. They are deleted when
# the session ends, so download the output as soon as it is saved.
DATA_DIR = Path("/content")
INPUT_PATH = DATA_DIR / "unique_statements_cleaned.csv"
OUTPUT_PATH = DATA_DIR / "embeddings_8b.npy"
assert INPUT_PATH.exists(), "Drag unique_statements_cleaned.csv into the Files panel first"

In [ ]:
import torch
from transformers import BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

assert torch.cuda.is_available(), "No GPU, switch the runtime type to GPU"
gpu = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu} ({gpu_mem_gb:.0f} GB)")

# The model is ~16 GB in half precision, which needs a 24 GB+ card (L4, A100).
# On a 16 GB T4, fall back to 8-bit weights (~8 GB), slightly lower fidelity.
if gpu_mem_gb >= 20:
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_kwargs = {"dtype": dtype}
    BATCH_SIZE = 32
    print(f"Full model in {dtype}")
else:
    model_kwargs = {"quantization_config": BitsAndBytesConfig(load_in_8bit=True)}
    BATCH_SIZE = 8
    print("Small GPU, loading in 8-bit")

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

statements = pd.read_csv(INPUT_PATH)["statement"].tolist()
print(f"Loaded {len(statements)} statements")

# First run downloads ~16 GB of weights, takes a few minutes
model = SentenceTransformer(MODEL_NAME, device="cuda", model_kwargs=model_kwargs)
model.max_seq_length = 256  # statements are short, this caps memory per batch

In [ ]:
# Same settings as src/embed.py (no prompt, no normalization) so results stay comparable
embeddings = model.encode(statements, show_progress_bar=True, batch_size=BATCH_SIZE, convert_to_numpy=True)
embeddings = embeddings.astype(np.float32)
print(f"Embeddings shape: {embeddings.shape}")

np.save(OUTPUT_PATH, embeddings)
print(f"Saved to {OUTPUT_PATH}, download it from the Files panel now")

## Optional: embed other text with the same model

For Sprint 4 nearest-neighbor checks, O*NET / ESCO descriptions must be embedded with this same model. Drag a CSV into the Files panel and set the file and column names below.

In [ ]:
EXTRA_CSV = ""  # e.g. "esco_skills.csv"
EXTRA_COLUMN = "description"

if EXTRA_CSV:
    texts = pd.read_csv(DATA_DIR / EXTRA_CSV)[EXTRA_COLUMN].fillna("").tolist()
    extra = model.encode(texts, show_progress_bar=True, batch_size=BATCH_SIZE, convert_to_numpy=True)
    out = DATA_DIR / f"{Path(EXTRA_CSV).stem}_embeddings_8b.npy"
    np.save(out, extra.astype(np.float32))
    print(f"Saved {extra.shape} to {out}")